# Multicollinearity Reduction (VIF Elimination)

Removes redundant features from `model_dataset.parquet` using iterative VIF
(Variance Inflation Factor) elimination, before hyperparameter tuning and final
training.

**Why VIF, not pairwise correlation:** pairwise correlation only catches
redundancy between two features at a time. VIF catches multivariate redundancy —
a feature that's well-explained by *several* others combined, even when its
correlation with any single one looks low. This matters here because many
features are variants of the same aggregate across different time windows
(1Y/3Y/6M/3M), which is exactly the kind of multi-feature redundancy pairwise
correlation misses.

**Why VIF is recomputed here rather than reused from `select_vars_refined.ipynb`:**
VIF is only meaningful relative to whichever feature set is currently under
consideration. The VIF values in `numeric_selection_metrics_full_df.xlsx` were
computed per-dataset *before* the tables were joined and filtered down to the
current feature set — they don't reflect the redundancy that exists in the
358-feature joined dataset today.

**Pipeline overview**
1. Load `model_dataset.parquet`
2. Compute VIF (pseudo-inverse method — fast, no per-feature OLS)
3. Iteratively drop the single highest-VIF feature above threshold, recompute, repeat
4. Save the reduced dataset & elimination log

**Output**
- `src/intermediate/model_dataset.parquet` — overwritten in place with reduced features
- `src/Variable-Selection/selected_features.csv` — updated to match
- `src/Variable-Selection/vif_elimination_log.csv` — full audit trail of what was dropped and why

**After this notebook runs:** re-run `02_hyperparameter_tuning.ipynb` and
`03_training_and_analysis.ipynb` — the feature set has changed, so both the search
space results and the final model need to be regenerated against the reduced dataset.

## 1. Imports & Configuration

In [10]:
import os
import pandas as pd
import numpy as np

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")
HPO_DIR  = os.path.join(DATA_ROOT, "HyperParameters", "LightGBM_v1")

os.makedirs(HPO_DIR, exist_ok=True)

PATH_MODEL_DATASET = os.path.join(DATA_ROOT, "ModelDataset", "model_dataset.parquet")
VAR_SELECTION_DIR = os.path.join(DATA_ROOT, "Variable-Selection")
os.makedirs(VAR_SELECTION_DIR, exist_ok=True)

PATH_SELECTED_FEATURES = os.path.join(VAR_SELECTION_DIR, "selected_features.csv")
PATH_ELIMINATION_LOG = os.path.join(VAR_SELECTION_DIR, "vif_elimination_log.csv")
PATH_CORR_MATRIX = os.path.join(VAR_SELECTION_DIR, "correlation_matrix.xlsx")
PATH_VIF_VALUES = os.path.join(VAR_SELECTION_DIR, "vif_values.xlsx")
PATH_SELECTION_METRICS = os.path.join(VAR_SELECTION_DIR, "numeric_selection_metrics_full_df.xlsx")

# Standard textbook threshold. Stricter: 5. Looser: 10 is the most common default
# (e.g. Kutner et al., "Applied Linear Statistical Models").
VIF_THRESHOLD = 10.0
CORR_PARTNER_THRESHOLD = 0.7

### Load Feature Predictive Scores

Loads the `points` score (0–6) computed in `select_vars_refined.ipynb` for every
numeric feature — a combined measure of AUC/Gini, t-test significance, Cohen's d,
logistic coefficient, and target correlation. This is used below to decide *which*
member of a correlated group to drop, not just to blindly follow VIF.

In [11]:
selection_metrics = pd.read_excel(PATH_SELECTION_METRICS)
feature_score = dict(zip(selection_metrics['variable'], selection_metrics['points']))

# Features with no score on record (shouldn't normally happen for numeric columns
# that survived Notebook 1's selection step) default to 0 — lowest priority,
# dropped first if involved in a redundant group.
missing_scores = [f for f, score in feature_score.items() if score is None or np.isnan(score)]
if missing_scores:
    print(f"WARNING: {len(missing_scores)} features have no selection score "
          f"on record — defaulting to score 0 (lowest priority): {missing_scores[:5]}...")
    for f in missing_scores:
        feature_score[f] = 0

print(f"Loaded predictive scores for {len(feature_score)} features.")

Loaded predictive scores for 1570 features.


## 2. Load Dataset

VIF only applies to numeric features. Categorical (`category`-dtype) columns,
`SK_ID_CURR`, and `TARGET` are set aside and always kept — they're never candidates
for elimination.

In [12]:
model_dataset = pd.read_parquet(PATH_MODEL_DATASET)

cat_cols = model_dataset.select_dtypes(include=['object', 'category']).columns.tolist()
time_indicator_cols = [c for c in model_dataset.columns  if 'IS_IN_' in c]
numeric_cols = [
    c for c in model_dataset.columns
    if c not in cat_cols + time_indicator_cols + ['SK_ID_CURR', 'TARGET']
]

print(f"Loaded dataset       : {model_dataset.shape}")
print(f"Numeric features (candidates for elimination): {len(numeric_cols)}")
print(f"Categorical features (always kept)           : {len(cat_cols)}")

Loaded dataset       : (307511, 218)
Numeric features (candidates for elimination): 179
Categorical features (always kept)           : 10


## 3. VIF Computation — Compute Once, Slice Per Iteration

**Why the original approach was slow:** correlation between two surviving
features never changes when a third feature is dropped — `corr(A, B)` is the
same value regardless of whether `C` is still in the dataset. Recomputing the
full pairwise correlation matrix from the raw 307K-row data on every iteration
(as the naive loop does) is pure wasted work — with ~358 columns, pandas'
`.corr()` does a row-by-row Cython pass over roughly `rows × cols²` cell pairs
*every single time*, which is where the multi-minute-per-iteration cost comes from.

**The fix:** compute the full correlation matrix **once**, using `numpy.corrcoef`
(BLAS-backed matrix multiplication — much faster than pandas' NaN-aware Cython
loop, and safe here since the data was already imputed with no `NaN`s remaining).
Every iteration of the elimination loop then just **slices out** the rows/columns
of whichever features currently remain, and re-runs `pinv()` on that shrinking
submatrix — a `pinv()` call on even the full 358×358 matrix takes milliseconds,
so hundreds of iterations become effectively free once the one-time correlation
matrix is built.

In [13]:
def build_full_correlation_matrix(df, feature_cols):
    """
    Compute the full pairwise correlation matrix ONCE using numpy (BLAS-backed,
    much faster than pandas' .corr() Cython loop). Returned as a DataFrame so it
    can be sliced by feature name in the elimination loop below.

    Assumes df has no NaN values (already imputed).
    """
    values = df[feature_cols].values.astype(np.float64)
    corr_array = np.corrcoef(values, rowvar=False)
    return pd.DataFrame(corr_array, index=feature_cols, columns=feature_cols)


def vif_from_correlation_slice(corr_df, feature_cols):
    """
    Compute VIF for a subset of features by slicing a precomputed full
    correlation matrix — no recomputation from raw data required.

    Returns
    -------
    pd.DataFrame with columns ['feature', 'VIF'], sorted descending by VIF.
    """
    sub_corr = corr_df.loc[feature_cols, feature_cols]
    inverse_matrix = np.linalg.pinv(sub_corr.values)
    vifs = np.diag(inverse_matrix)
    vif_df = pd.DataFrame({'feature': feature_cols, 'VIF': vifs})
    return vif_df.sort_values('VIF', ascending=False).reset_index(drop=True)

## 4. Iterative VIF Elimination — Protecting Strong Predictors

At each step: slice the precomputed correlation matrix to find the highest-VIF
feature, then look at which *other remaining features* it's strongly correlated
with (`|corr| >= CORR_PARTNER_THRESHOLD`). Among the flagged feature and its
correlated partners, the one with the **lowest predictive score** is dropped —
not necessarily the flagged feature itself.

**Why this matters:** VIF only measures redundancy among predictors — it has no
concept of which predictor is actually useful. Without this protection, a strong
feature (e.g. `EXT_SOURCES_1`) could be dropped purely because it happened to
have the highest VIF at some iteration, while several weaker correlated variants
survive. This step ensures VIF decides *where* redundancy exists, while the
predictive score decides *what* gets sacrificed from within that redundant group.

If the flagged feature has no correlated partners above threshold (i.e. its high
VIF comes from diffuse multivariate redundancy across many weakly-correlated
features, not one dominant pair), there's no alternative to swap in — it gets
dropped as originally flagged.

In [14]:
from sklearn.impute import SimpleImputer

remaining_features = numeric_cols.copy()
elimination_log = []
iteration = 0
protected_swaps = 0

imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(model_dataset[remaining_features]),
    columns=remaining_features,
)

print(f"Building correlation matrix once for {len(remaining_features)} features...")
full_corr_matrix = build_full_correlation_matrix(X_imputed, remaining_features)
print("Correlation matrix built. Starting elimination.\n")
print(f"VIF threshold: {VIF_THRESHOLD}  |  Partner correlation threshold: {CORR_PARTNER_THRESHOLD}\n")

while True:
    vif_df = vif_from_correlation_slice(full_corr_matrix, remaining_features)
    max_vif_row = vif_df.iloc[0]

    if max_vif_row['VIF'] <= VIF_THRESHOLD:
        print(f"Stopped: max VIF = {max_vif_row['VIF']:.2f} "
              f"({max_vif_row['feature']}) is at or below threshold.")
        break

    flagged_feature = max_vif_row['feature']
    flagged_vif = max_vif_row['VIF']

    # Find features strongly correlated with the flagged feature among those remaining
    partner_corrs = full_corr_matrix.loc[flagged_feature, remaining_features].abs()
    partners = partner_corrs[
        (partner_corrs >= CORR_PARTNER_THRESHOLD) & (partner_corrs.index != flagged_feature)
    ].index.tolist()

    drop_pool = [flagged_feature] + partners

    # FIX: drop whichever member of the redundant group has the LOWEST predictive
    # score — protects strong predictors even if they're the technically-highest-VIF one
    feature_to_drop = min(drop_pool, key=lambda f: feature_score.get(f, 0))

    # if feature with the highest VIF is not the one with the lowest predictive score, count it as a "protected swap"
    was_protected_swap = (feature_to_drop != flagged_feature)
    if was_protected_swap:
        protected_swaps += 1

    elimination_log.append({
        'iteration': iteration,
        'flagged_feature': flagged_feature,
        'flagged_feature_vif': flagged_vif,
        'flagged_feature_score': feature_score.get(flagged_feature, 0),
        'dropped_feature': feature_to_drop,
        'dropped_feature_score': feature_score.get(feature_to_drop, 0),
        'was_protected_swap': was_protected_swap,
        'features_remaining_after': len(remaining_features) - 1,
    })

    remaining_features.remove(feature_to_drop)
    iteration += 1

    if iteration % 10 == 0:
        print(f"  [{iteration:03d}] dropped '{feature_to_drop}' "
              f"(score={feature_score.get(feature_to_drop, 0)}) — "
              f"{len(remaining_features)} features remain")

print(f"\nElimination complete: {iteration} features dropped "
      f"({protected_swaps} were protected swaps — a weaker correlated feature "
      f"was dropped instead of the flagged one), {len(remaining_features)} features remain.")

Building correlation matrix once for 179 features...
Correlation matrix built. Starting elimination.

VIF threshold: 10.0  |  Partner correlation threshold: 0.7

Stopped: max VIF = 9.50 (CREDITCARD_DRAWING_TO_PYMNT_RATIO_MAX_LAST6M) is at or below threshold.

Elimination complete: 0 features dropped (0 were protected swaps — a weaker correlated feature was dropped instead of the flagged one), 179 features remain.


## 5. Diagnostics

In [15]:
final_vif = vif_from_correlation_slice(full_corr_matrix, remaining_features)

print(f"Features before elimination : {len(numeric_cols)}")
print(f"Features after elimination  : {len(remaining_features)}")
print(f"Features dropped            : {len(numeric_cols) - len(remaining_features)}")
print(f"Max VIF after elimination   : {final_vif['VIF'].max():.2f}")
print(f"Mean VIF after elimination  : {final_vif['VIF'].mean():.2f}")

elimination_log_df = pd.DataFrame(elimination_log)

if len(elimination_log_df) > 0:
    # FIX: column is 'flagged_feature_vif' in the new log schema, not 'vif_at_removal'
    print("\nTop 15 highest-VIF drops (by the flagged feature's VIF at that step):")
    print(
        elimination_log_df
        .sort_values('flagged_feature_vif', ascending=False)
        .head(15)[['flagged_feature', 'flagged_feature_vif', 'dropped_feature']]
        .to_string(index=False, float_format='{:.1f}'.format)
    )

protected_examples = (
    elimination_log_df[elimination_log_df['was_protected_swap']]
    if len(elimination_log_df) > 0 else pd.DataFrame()
)
if len(protected_examples) > 0:
    print(f"\n{len(protected_examples)} protected swaps occurred — "
          f"strong features that were saved by dropping a weaker correlated partner instead:")
    print(
        protected_examples[
            ['flagged_feature', 'flagged_feature_score', 'dropped_feature', 'dropped_feature_score']
        ]
        .head(10)
        .to_string(index=False)
    )
else:
    print("\nNo protected swaps occurred — every dropped feature was the flagged "
          "highest-VIF feature with no stronger correlated partner to protect.")

Features before elimination : 179
Features after elimination  : 179
Features dropped            : 0
Max VIF after elimination   : 9.50
Mean VIF after elimination  : 3.76

No protected swaps occurred — every dropped feature was the flagged highest-VIF feature with no stronger correlated partner to protect.


Additional Features from eliminated protected features

In [16]:
#additional_features_from_protected = protected_examples[protected_examples['dropped_feature_score']>=4].reset_index()

## 6. Save Reduced Dataset & Audit Trail

`model_dataset.parquet` is overwritten in place with the reduced feature set —
this is a pipeline stage, so downstream notebooks (`02_hyperparameter_tuning.ipynb`,
`03_training_and_analysis.ipynb`) don't need to know or care which stage last wrote
this file, only that it contains the current feature set. Categorical columns and
`TARGET` are untouched.

`vif_elimination_log.csv` is the audit trail — which feature was dropped at each
step and its VIF at the moment of removal, useful if you ever need to justify or
revisit why a specific feature was excluded.

In [ ]:
final_feature_cols = remaining_features + cat_cols + time_indicator_cols
reduced_dataset = model_dataset[['SK_ID_CURR'] + final_feature_cols + ['TARGET']]

full_corr_matrix.to_excel(PATH_CORR_MATRIX, index=True)
print(f"Saved correlation matrix → {PATH_CORR_MATRIX}")

final_vif.to_excel(PATH_VIF_VALUES, index=False)
print(f"Saved final VIF values → {PATH_VIF_VALUES}")

reduced_dataset.to_parquet(PATH_MODEL_DATASET, index=False)
print(f"Saved reduced model_dataset.parquet ({reduced_dataset.shape}) → {PATH_MODEL_DATASET}")

pd.Series(final_feature_cols).to_csv(PATH_SELECTED_FEATURES, index=False)
print(f"Saved updated feature list → {PATH_SELECTED_FEATURES}")

elimination_log_df.to_csv(PATH_ELIMINATION_LOG, index=False)
print(f"Saved elimination log ({len(elimination_log_df)} rows) → {PATH_ELIMINATION_LOG}")

Saved reduced model_dataset.parquet ((307511, 218)) → a:\Dominic\My Projects\Home Credit\data\ModelDataset\model_dataset.parquet
Saved updated feature list → a:\Dominic\My Projects\Home Credit\data\Variable-Selection\selected_features.csv
Saved elimination log (374 rows) → a:\Dominic\My Projects\Home Credit\data\Variable-Selection\vif_elimination_log.csv


## 7. Cleanup

In [9]:
del model_dataset, X_imputed, reduced_dataset
import gc; gc.collect()
print("All DataFrames released.")

print("\nNext step: re-run 02_hyperparameter_tuning.ipynb and "
      "03_training_and_analysis.ipynb against the reduced feature set.")

All DataFrames released.

Next step: re-run 02_hyperparameter_tuning.ipynb and 03_training_and_analysis.ipynb against the reduced feature set.
